# PronunciaBench — ByT5 CMUdict G2P (Kaggle)

## Setup
1. Enable GPU: Settings → Accelerator → GPU T4/P100
2. Run all cells. The script auto-detects GPU and falls back to smaller batches on OOM.
3. Results are saved to `kaggle_output/`.

In [ ]:
# Clone repo and install
!git clone https://github.com/NostalgiaFleeting/PronunciaBench.git
%cd PronunciaBench
%cd experiment/byt5-cmudict 2>/dev/null || true
!pip install -e ".[dev]" -q
!pip install requests -q

In [ ]:
# GPU diagnostics
import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GB')
    print(f'Bf16 supported: {props.major >= 8}')
else:
    print('No GPU found — running on CPU (will be very slow)')

In [ ]:
# Import and run CMUdict data preparation
import sys; sys.path.insert(0, 'scripts')
from cmudict_import import fetch_cmudict, split_dataset, audit_leakage
print('CMUdict import ready')

In [ ]:
# Run the experiment
import subprocess, sys
result = subprocess.run([
    sys.executable, 'scripts/run_byt5_cmudict.py',
    '--data-dir', 'data/experiment',
    '--epochs', '3',
    '--batch-size', '32',
    '--gradient-accumulation', '2',
    '--seed', '42',
], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-2000:])
print('Exit code:', result.returncode)

In [ ]:
# Print final results
import json
from pathlib import Path
results_files = list(Path('experiments/byt5-cmudict-001').glob('**/results.json'))
if results_files:
    d = json.load(open(results_files[-1]))
    print(json.dumps(d, indent=2))
else:
    print('No results found — check training output above')